# 38. Is our CatBoost simply undertrained?

**One variable against ledger row 70** (`cat_raw`, CV 0.961420): the number of boosting
iterations. Same raw 12-column frame, same 3 native categoricals, same NaN handling, same
`learning_rate=0.05`, same seed, same folds, same thread count, every other CatBoost
parameter left at its library default exactly as `36_no_encoder_views.ipynb` left them.

## Why this is being run, and why it is not the argument row 62 got wrong

Rows 62 to 65 swept XGBoost depth on an argument that the knob "had never been fitted to
XGBoost". The rule in this repo assigns that a low prior, the rule was right, and the sweep
was a bracketed null. So a budget-was-inherited argument on its own is worth very little here
and has a losing record.

**This one has an out-of-sample check attached, which that one did not.** On the identical
fold split, verified bit-identical:

| on the same raw frame | ours | public |
|---|---|---|
| LightGBM | 0.963464 | 0.966381 |
| XGBoost | 0.964218 | 0.967042 |
| CatBoost | **0.961420** | **0.968405** |

Ours is the only pipeline in which CatBoost finishes **last**. Publicly it finishes first and
beats its own XGBoost by +0.0014. A family that is best in someone else's hands and worst in
ours is not a statement about the family, and the deficit is 0.0070, which is five times the
largest gap in the other two rows.

The suspect is `learning_rate=0.05` with `n_estimators=2000`, inherited from the LightGBM
budget convention when row 26 was written on 2026-08-17 and never once fitted to CatBoost.
CatBoost grows **oblivious** trees, which use one split condition per level across the whole
level. They are deliberately weaker per tree than LightGBM's leaf-wise trees, so the same
iteration count buys materially less capacity. A budget borrowed from LightGBM is exactly the
thing that would under-train them, and it would do it silently.

## The design: one fit per fold, the whole curve read off it

Gradient boosting is sequential, so trees 1 to k of a 12,000-iteration model **are** the
model you would have got by asking for k iterations. One fit per fold at `N_MAX`, then
`predict_proba(..., ntree_end=k)` at every checkpoint. The whole curve costs one fit.

Two things fall out of that, and both are why the design was chosen:

- **A free reproduction check.** The `k=2000` checkpoint must reproduce row 70's 0.961420 to
  floating-point noise, because it is the same model. If it does not, something in the
  environment moved and every number below is void.
- **No `eval_set` anywhere.** The model never receives the validation fold in any form, so
  there is no early stopping, no `use_best_model`, and no surface for the validation fold to
  influence the fit. Public notebooks on this competition early-stop on the fold their OOF is
  scored on, which is a mild optimism this repo has declined since row 35.

## Version 1 of this notebook failed its own check, and the reason is worth the section

Version 1 was pushed on 2026-08-21, ran the bench, and **stopped itself** on the `ntree_end`
contract at `max |direct - sliced| = 2.214e-01`. It had passed the identical check locally at
smoke scale. Five hours of CPU were not spent, which is what the check is for.

The cause is not `ntree_end`. **CatBoost silently auto-selects two structural parameters from
the value of `iterations`:**

| iterations | `leaf_estimation_iterations` | `max_ctr_complexity` |
|---|---|---|
| up to 150 | 1 | 1 |
| 200 and above | 10 | 4 |

Verified by fitting and reading `get_all_params()`, and constant from 200 through at least
4000. The bench compared a 100-iteration model against a 200-iteration model sliced to 100.
Those straddle the threshold, so they are **different model families**, and the check correctly
said so. The smoke run compared 25 against 50, both below the threshold, and passed. Both
results were right; the check was asking the wrong pair.

**The real hazard is larger than this notebook.** Sweeping `iterations` on default CatBoost is
not a one-variable experiment at all: it moves the boosting budget, the leaf estimation, and
the depth of category combinations together, and it does it without a warning. Any ledger row
that compared two CatBoost budgets across that threshold would have been comparing three
changes and calling it one.

Both parameters are therefore **pinned** here at the values CatBoost itself picks at 2000
iterations, which is what row 70 ran with. That makes `iterations` genuinely the only variable
and leaves the k=2000 arm an exact reproduction of row 70. The bench pair is now 200 from 400,
both above the threshold.

## The checkpoints, pre-registered before the run

`2000, 3000, 4000, 6000, 8000, 10000, 12000`.

Fixed here so that reading the best one off the curve afterwards is a sweep over a declared
grid, which is what rows 62 to 65 did with depth, and not a search over an open set.

## The prediction, written before the run

**I predict the curve is still climbing at 2000 and that CatBoost here is undertrained**,
worth +0.002 to +0.005 by 12,000, landing somewhere in 0.9635 to 0.9665.

I also predict that **this does not close the gap on its own**. Public CatBoost on a raw frame
is 0.968405, and if the budget were the whole story the curve would have to reach it. The
remainder is most likely the feature block their views carry and ours do not: composition
ratios, the generator's slack term, imputed copies alongside the NaN columns. That is the open
item added to this repo on 2026-08-21 and it is a different run.

**The honest case against running this at all.** Rows 46 to 49, 50 to 51, 53 to 54 and 62 to 65
were four consecutive knob nulls, and the six-point rule went seven for seven while my
mechanism arguments went one for four. If the curve is flat past 2000 then CatBoost was already
converged, the family inversion is entirely a feature-set story, and this row is the fifth null.
That would still be worth knowing, because it would point the next run squarely at the features.

## What this decides

Nothing about the stack. It writes one member vector per checkpoint. Membership is a separate
notebook and a separate ledger row, per the rule that has held since row 24. No submission csv.

In [ ]:
# One flag. The run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42

# Row 70's configuration, held fixed. Only N_EST becomes a swept quantity.
LR = 0.05
PROBE_FOLD = 0

# PINNED, and this is the whole reason version 1 of this notebook failed. CatBoost
# auto-selects both of these from `iterations`, switching from 1 and 1 to 10 and 4
# somewhere between 150 and 200. Row 70 ran at 2000 iterations and therefore got 10 and 4.
# Pinning them at those values makes this sweep one variable instead of three, and leaves
# the k=2000 arm an exact reproduction of row 70.
LEAF_EST_ITERS = 10
MAX_CTR_COMPLEXITY = 4

# Above the auto-tune threshold, so the bench pair does not straddle it. Version 1 used
# 200 with a half of 100, which compared a 1-and-1 model against a slice of a 10-and-4
# model and reported the ntree_end contract broken when the contract was fine.
BENCH_EST = 400

# The pre-registered grid. N_MAX is the single fit; the rest are read off it.
CHECKPOINTS = [2000, 3000, 4000, 6000, 8000, 10000, 12000]
N_MAX = max(CHECKPOINTS)

# Row 70 ran on Kaggle at the library default thread count. Matching it is part of
# matching it.
THREADS = -1

# Hard guard. Kaggle allows 12h on CPU; the bench stage projects the full run and stops
# here rather than after five hours of a run that was never going to finish.
MAX_HOURS = 9.0

# The row this changes one variable against.
BASELINE_NAME, BASELINE_CV, BASELINE_ROW = "cat_raw", 0.961420, "row 70 cat_raw"
EXPECTED_FOLD_SHA = "ec282b0968059676"

print(f"SMOKE = {SMOKE}   lr {LR}   checkpoints {CHECKPOINTS}")
print(f"one fit per fold at {N_MAX:,} iterations, curve read via ntree_end")

## Stage 1. Data, folds, leak checklist

Lifted from `36_no_encoder_views.ipynb` unchanged, because this notebook must build the
identical frame to the row it is compared against. The fold checksum is checked before
anything trains, since a misaligned out-of-fold vector blends silently and wrongly.

In [ ]:
import gc
import hashlib
import time
from pathlib import Path

import catboost as cb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")
print(f"catboost {cb.__version__}, pandas {pd.__version__}, numpy {np.__version__}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    CHECKPOINTS = [50, 100, 200]
    N_MAX, BENCH_EST = 200, 50
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

## Stage 2. The representation, and the leak argument

Identical to row 70's frame. **Nothing here touches the target**, which is the whole leak
argument, and it is asserted rather than described: the feature frame must be bit-identical
under a permutation of `y`.

`DataFrame.equals` treats NaN in the same position as equal, which plain `==` does not. On
pandas 3.0 `astype(str)` preserves NA, so an `==` comparison silently reports every missing
cell as unequal. That is the same version trap that eats target encodings, met here in a
check rather than in a feature, and it is why the comparison below is written this way.

In [ ]:
def frame(df):
    X = df[COLS].copy()
    for c in CAT:
        # CatBoost takes categoricals as strings and will not accept NaN in them.
        X[c] = X[c].astype(object).fillna("__missing__").astype(str)
    return X


X_cb = frame(train)
X_test_cb = frame(test)
CAT_IDX = [X_cb.columns.get_loc(c) for c in CAT]

rng = np.random.default_rng(0)
train_perm = train.copy()
train_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
pure = frame(train_perm).equals(X_cb)
print(f"feature frame is a pure function of the features, not of y: {pure}")
print(f"target columns present in the feature set: "
      f"{[c for c in X_cb.columns if c in ('id', TARGET)]}")
print(f"{len(COLS)} raw columns, {len(CAT)} categorical handed to CatBoost as cat_features,")
print(f"{len(COLS) - len(CAT)} numeric columns left NUMERIC so ordered splits remain available.")
print(f"cat_features indices {CAT_IDX}")
CLEAN = pure and LEAK_OK
print()
print(f"leak checks: {'PASS' if CLEAN else 'FAILED'}")

## Stage 3. Bench, determinism, and the projection

The bench trains the same small configuration twice and requires bit-identical output. A
comparison between two rows means nothing unless the pipeline is deterministic, and this repo
found on 2026-08-04 that assuming it is, is how you get a wrong ledger.

It then projects the full run from measured seconds per iteration and stops here rather than
after five hours of a run that was never going to finish.

In [ ]:
def make_cat(n_est):
    # Row 70's configuration, with the two auto-tuned parameters PINNED at the values
    # CatBoost itself selects at 2000 iterations. See the header: leaving them implicit
    # makes `iterations` silently change three things instead of one.
    #
    # No eval_set, no early stopping, no use_best_model. The model never sees the
    # validation fold, so there is nothing for it to overfit to.
    return cb.CatBoostClassifier(
        iterations=n_est, learning_rate=LR, random_seed=SEED,
        leaf_estimation_iterations=LEAF_EST_ITERS, max_ctr_complexity=MAX_CTR_COMPLEXITY,
        thread_count=THREADS, allow_writing_files=False, verbose=0,
    )


def fit_fold(fold, n_est):
    tr = np.where(folds != fold)[0]
    t0 = time.time()
    m = make_cat(n_est)
    m.fit(X_cb.iloc[tr], y[tr], cat_features=CAT_IDX)
    return m, time.time() - t0


def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


va0 = np.where(folds == PROBE_FOLD)[0]
m1, t1 = fit_fold(PROBE_FOLD, BENCH_EST)
p1 = m1.predict_proba(X_cb.iloc[va0])[:, 1]
m2, _ = fit_fold(PROBE_FOLD, BENCH_EST)
p2 = m2.predict_proba(X_cb.iloc[va0])[:, 1]
drift = float(np.max(np.abs(p1 - p2)))
print(f"bench {BENCH_EST} iters: AUC {roc_auc_score(y[va0], p1):.6f} in {hhmm(t1)}, "
      f"repeat drift {drift:.3e} {'OK' if drift == 0.0 else 'NOT DETERMINISTIC'}")

# The ntree_end contract, checked at bench scale before four hours are spent on it:
# predicting with ntree_end=k from a longer model must equal a k-iteration model exactly.
half = max(2, BENCH_EST // 2)
m_half, _ = fit_fold(PROBE_FOLD, half)
p_half_direct = m_half.predict_proba(X_cb.iloc[va0])[:, 1]
p_half_sliced = m1.predict_proba(X_cb.iloc[va0], ntree_end=half)[:, 1]
slice_drift = float(np.max(np.abs(p_half_direct - p_half_sliced)))
print(f"ntree_end contract at k={half}: max |direct - sliced| = {slice_drift:.3e} "
      f"{'OK' if slice_drift == 0.0 else 'BROKEN, the curve below would be meaningless'}")
NTREE_OK = slice_drift == 0.0
del m2, m_half
gc.collect()
m_pred = m1

# Prediction is NOT free here and the first version of this projection ignored it.
# Reading 7 checkpoints means predicting 2000+3000+...+12000 = 45,000 tree-evaluations
# worth of work per fold, over the validation fold plus the whole test set, which is
# several times one ordinary predict. Measured rather than guessed.
t0 = time.time()
_ = m_pred.predict_proba(X_cb.iloc[va0], ntree_end=BENCH_EST)[:, 1]
_ = m_pred.predict_proba(X_test_cb, ntree_end=BENCH_EST)[:, 1]
per_pred_iter = (time.time() - t0) / BENCH_EST

per_iter = t1 / BENCH_EST
train_secs = per_iter * N_MAX * 5
# Prediction scales with the number of trees actually walked, so it is the SUM of the
# checkpoints and not the maximum.
pred_secs = per_pred_iter * sum(CHECKPOINTS) * 5
projected = train_secs + pred_secs
print(f"\nprojected training:   {hhmm(train_secs)} for 5 folds x {N_MAX:,} iterations")
print(f"projected prediction: {hhmm(pred_secs)} for {len(CHECKPOINTS)} checkpoints "
      f"({sum(CHECKPOINTS):,} tree-evaluations per fold, train and test)")
print(f"projected TOTAL:      {hhmm(projected)}")
GO = projected < MAX_HOURS * 3600 or SMOKE
print("within budget" if GO else
      f"OVER the {MAX_HOURS}h guard, not starting. Drop checkpoints or N_MAX and re-push.")

del m1, m_pred
gc.collect()

## Stage 4. The run

One fit per fold. Every checkpoint read off it. Out-of-fold and test vectors written per
checkpoint so that `39` can test membership without refitting anything.

In [ ]:
assert CLEAN, "leak checks failed"
assert NTREE_OK, "ntree_end contract failed"
assert GO, "over the time guard"
if not SMOKE:
    assert ALIGNED, "fold sha mismatch"

oof = {k: np.zeros(len(train)) for k in CHECKPOINTS}
tst = {k: np.zeros((5, len(test))) for k in CHECKPOINTS}
per_fold = {k: [] for k in CHECKPOINTS}

t_start = time.time()
for f in range(5):
    m, secs = fit_fold(f, N_MAX)
    va = np.where(folds == f)[0]
    line = []
    for k in CHECKPOINTS:
        pv = m.predict_proba(X_cb.iloc[va], ntree_end=k)[:, 1]
        oof[k][va] = pv
        tst[k][f] = m.predict_proba(X_test_cb, ntree_end=k)[:, 1]
        a = float(roc_auc_score(y[va], pv))
        per_fold[k].append(a)
        line.append(f"{k}:{a:.6f}")
    del m
    gc.collect()
    done = time.time() - t_start
    print(f"fold {f} in {hhmm(secs)}  " + "  ".join(line))
    print(f"         elapsed {hhmm(done)}, about {hhmm(done / (f + 1) * (4 - f))} left")

print(f"\nall folds done in {hhmm(time.time() - t_start)}")

In [ ]:
cv = {k: float(np.mean(per_fold[k])) for k in CHECKPOINTS}
sd = {k: float(np.std(per_fold[k])) for k in CHECKPOINTS}

print("The curve. k=2000 is row 70 refit, not a treatment.\n")
print(f"{'iterations':>11} {'CV':>10} {'sd':>9} {'vs k=2000':>11} {'folds won':>10}")
base = np.array(per_fold[CHECKPOINTS[0]])
for k in CHECKPOINTS:
    d = np.array(per_fold[k]) - base
    w = "" if k == CHECKPOINTS[0] else f"{int((d > 0).sum())}/5"
    v = "" if k == CHECKPOINTS[0] else f"{d.mean():+11.6f}"
    print(f"{k:>11,} {cv[k]:10.6f} {sd[k]:9.6f} {v:>11} {w:>10}")

repro = cv[CHECKPOINTS[0]] - BASELINE_CV
print(f"\nreproduction of {BASELINE_ROW}: {cv[CHECKPOINTS[0]]:.6f} vs {BASELINE_CV:.6f}"
      f"  delta {repro:+.2e}")
if SMOKE:
    print("SMOKE: this did NOT run at full size, so the reproduction check DID NOT RUN.")
else:
    print("REPRODUCED" if abs(repro) < 1e-4 else "FAILED - every number above is void")

best_k = max(CHECKPOINTS, key=lambda k: cv[k])
d = np.array(per_fold[best_k]) - base
print(f"\nbest checkpoint {best_k:,} at {cv[best_k]:.6f}")
print(f"  paired vs k=2000: {d.mean():+.6f}, sd {d.std(ddof=1):.6f}, "
      f"{int((d > 0).sum())}/5 folds")
print(f"  is the optimum interior to the grid? "
      f"{'yes, bracketed' if best_k not in (CHECKPOINTS[0], CHECKPOINTS[-1]) else 'NO, it is at an edge of the grid'}")
print(f"\nfor context on the same raw frame:")
print(f"  ours   lgb_raw 0.963464   xgb_raw 0.964218   cat_raw 0.961420")
print(f"  public lgb     0.966381   xgb     0.967042   cat     0.968405")

In [ ]:
# A smoke vector that reached artifacts/oof unprefixed would be indistinguishable from a
# real member and would blend silently. The prefix is what .gitignore keys on too.
pre = "SMOKE_" if SMOKE else ""
for k in CHECKPOINTS:
    np.save(OUT / f"{pre}cat_raw_n{k}_oof.npy", oof[k])
    np.save(OUT / f"{pre}cat_raw_n{k}_test.npy", tst[k].mean(axis=0))
    print(f"wrote {pre}cat_raw_n{k}_oof.npy, {pre}cat_raw_n{k}_test.npy")

print("\nledger lines:")
for k in CHECKPOINTS:
    print(f"  name    cat_raw_n{k}\n  cv_mean {cv[k]:.6f}\n  cv_std  {sd[k]:.6f}")
print(f"\n  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is decided in a separate notebook, which changes")
print("a different variable and gets its own ledger row.")